In [2]:
import pandas as pd
import numpy as np
import import_ipynb
import pickle
from decouple import Config, RepositoryEnv
import pymongo

In [3]:
data = pd.read_csv("../recipe_data/RAW_recipes.csv")
data.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [4]:
df2 = pd.read_pickle('/recipe_data/ingr_map.pkl')
df2.head()

,raw_ingr,raw_words,processed,len_proc,replaced,count,id
0,"medium heads bibb or red leaf lettuce, washed,...",13,"medium heads bibb or red leaf lettuce, washed,...",73,lettuce,4507,4308
1,mixed baby lettuces and spring greens,6,mixed baby lettuces and spring green,36,lettuce,4507,4308
2,romaine lettuce leaf,3,romaine lettuce leaf,20,lettuce,4507,4308
3,iceberg lettuce leaf,3,iceberg lettuce leaf,20,lettuce,4507,4308
4,red romaine lettuce,3,red romaine lettuce,19,lettuce,4507,4308


In [5]:
ingr = df2[["replaced", "id"]]
ingr = ingr.drop_duplicates()

In [51]:
config = Config(RepositoryEnv("../../.env"))
client = pymongo.MongoClient(config("MONGODB_URL"))
db = client["recipe_data"]



In [7]:
upload_recipes = data.drop(columns=["contributor_id","submitted"])
upload_recipes.head()

,name,id,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [8]:
upload_recipes["name"] = upload_recipes['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
upload_recipes.iloc[452]["name"]

'magic cookie bars'

In [9]:
# MIGRATE RECIPES TO MONGODB

#recipe_col = db["recipes"]

#for j, row in upload_recipes.iterrows():
#    recipe_col.insert_one({
#        "name": row['name'], 
#        "id": row.id, 
#        "minutes": row.minutes,
#        "tags": row.tags,
#        "nutrition": row.nutrition,
#        "n_steps": row.n_steps,
#        "steps": row.steps,
#        "description": row.description,
#        "ingredients": row.ingredients,
#        "n_ingredients": row.n_ingredients,
#        })

In [10]:
# EXTRACTING ITEM IDS & RATINGS PER USER

pp_user = pd.read_csv("PP_users.csv")
print(pp_user.head())
pp_user["items"] = pp_user["items"].apply(lambda x:x[1:-1]).str.split(", ").apply(lambda x: [int(y) for y in x])

itemList_byUsers_sum = pp_user["items"].apply(lambda x: len(x)).sum()
print(itemList_byUsers_sum )

pp_user["ratings"] = pp_user["ratings"].apply(lambda x:x[1:-1]).str.split(", ").apply(lambda x: [float(y) for y in x])
rateList_byUsers_sum = pp_user["ratings"].apply(lambda x: len(x)).sum()
print(rateList_byUsers_sum)




   u                                         techniques  \
0  0  [8, 0, 0, 5, 6, 0, 0, 1, 0, 9, 1, 0, 0, 0, 1, ...   
1  1  [11, 0, 0, 2, 12, 0, 0, 0, 0, 14, 5, 0, 0, 0, ...   
2  2  [13, 0, 0, 7, 5, 0, 1, 2, 1, 11, 0, 1, 0, 0, 1...   
3  3  [498, 13, 4, 218, 376, 3, 2, 33, 16, 591, 10, ...   
4  4  [161, 1, 1, 86, 93, 0, 0, 11, 2, 141, 0, 16, 0...   

                                               items  n_items  \
0  [1118, 27680, 32541, 137353, 16428, 28815, 658...       31   
1  [122140, 77036, 156817, 76957, 68818, 155600, ...       39   
2  [168054, 87218, 35731, 1, 20475, 9039, 124834,...       27   
3  [163193, 156352, 102888, 19914, 169438, 55772,...     1513   
4  [72857, 38652, 160427, 55772, 119999, 141777, ...      376   

                                             ratings  n_ratings  
0  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, ...         31  
1  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...         39  
2  [3.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 5.0, ...    

In [58]:
# EXTRACTING INGREDIENT IDS

pp_rec = pd.read_csv("PP_recipes.csv")


pp_rec_fields = pp_rec[["id", "ingredient_ids"]].copy()
pp_rec_fields["ingredient_ids"] = pp_rec_fields["ingredient_ids"].apply(lambda x: x[1:-1]).str.split(", ").apply(lambda x: [int(y) for y in x])

pp_rec_fields.head()



In [12]:
# ORGANIZE ALL RATINGS BY RECIPE

recipe_ratings = {}
item_rate = pp_user[["items", "ratings"]]
for i in item_rate.iterrows():
    item_list = i[1]["items"]
    ratings_list = i[1]["ratings"]
    for j in range(len(item_list)):
        if item_list[j] not in recipe_ratings:
            recipe_ratings[item_list[j]] = [ratings_list[j]]
        else:
            recipe_ratings[item_list[j]].append(ratings_list[j])



In [13]:
# CREATING ESSENTIAL FIELDS WITH ALL RECIPES

recipe_stats = data[["id","minutes","n_steps", "n_ingredients"]]
recipe_stats = recipe_stats.merge(pp_rec_fields, on="id", how="left")
recipe_stats["id"]             = recipe_stats["id"].astype(int)
recipe_stats["minutes"]        = recipe_stats["minutes"].astype(int)
recipe_stats["n_steps"]        = recipe_stats["n_steps"].astype(int)
recipe_stats["n_ingredients"]  = recipe_stats["n_ingredients"].astype(int)
recipe_stats["ingredient_ids"] = recipe_stats["ingredient_ids"].apply(lambda x: x if isinstance(x, list) else [])
recipe_stats["ratings"]        = recipe_stats["id"].map(recipe_ratings).apply(lambda x: x if isinstance(x, list) else [])
recipe_stats.head()

,id,minutes,n_steps,n_ingredients,ingredient_ids,ratings
0,137739,55,11,7,"[7933, 4694, 4795, 3723, 840, 5006, 6270]","[5.0, 5.0]"
1,31490,30,9,6,"[5481, 6324, 2499, 4717, 6276, 1170]",[4.0]
2,112140,130,6,13,[],"[5.0, 5.0, 4.0]"
3,59389,45,11,11,"[1170, 4918, 6426, 5185, 7099, 5006, 6009, 627...","[4.0, 5.0]"
4,44061,190,5,8,[],"[5.0, 5.0]"


In [14]:
# BAYESIAN FOR ALL RATINGS

rating_len = recipe_stats["ratings"].apply(lambda x: len(x))
rating_tot = recipe_stats["ratings"].apply(lambda x: sum(x))
rating_avg = recipe_stats["ratings"].apply(lambda x: 0 if len(x) == 0 else sum(x) / len(x))
rating_tot_avg = rating_avg.mean()
rating_len_avg = rating_len.mean()
print(rating_tot_avg, rating_len_avg)

recipe_stats["bayesian"] = recipe_stats["ratings"].apply(lambda x: (sum(x) + rating_tot_avg * rating_len_avg) / (len(x) + rating_len_avg))
recipe_stats.head()


1.776444144205202 1.7134352456645527


,id,minutes,n_steps,n_ingredients,ingredient_ids,ratings,bayesian
0,137739,55,11,7,"[7933, 4694, 4795, 3723, 840, 5006, 6270]","[5.0, 5.0]",3.512603
1,31490,30,9,6,"[5481, 6324, 2499, 4717, 6276, 1170]",[4.0],2.595906
2,112140,130,6,13,[],"[5.0, 5.0, 4.0]",3.616009
3,59389,45,11,11,"[1170, 4918, 6426, 5185, 7099, 5006, 6009, 627...","[4.0, 5.0]",3.243310
4,44061,190,5,8,[],"[5.0, 5.0]",3.512603


In [50]:
bay = recipe_stats[["id", "bayesian"]]
bay = bay.sort_values(by="bayesian", ascending=False)
bay_dict = bay[:10000].set_index("id")["bayesian"].to_dict()

with open('data.pkl', 'wb') as file:
    pickle.dump(bay_dict, file)

bay_dict

{31024: 4.888896146637364,
 81311: 4.878092117385156,
 113588: 4.866958490703674,
 63482: 4.86130381277613,
 84680: 4.859613375767667,
 82100: 4.853156278730203,
 153429: 4.84738147736143,
 361: 4.839773917864575,
 96770: 4.836167564075291,
 6682: 4.834833042997736,
 45374: 4.831931041973422,
 75785: 4.8311594554894945,
 113937: 4.830752518582028,
 130399: 4.829480336367656,
 35495: 4.827028373915179,
 141365: 4.827028373915179,
 164559: 4.825835511766508,
 137287: 4.825835511766508,
 163792: 4.823864454441163,
 39272: 4.822316975351482,
 36523: 4.821363643682276,
 172390: 4.820767526620506,
 118206: 4.820164883038708,
 71071: 4.819641940890756,
 80022: 4.819641940890756,
 97466: 4.817661967704864,
 54631: 4.817539554201845,
 68338: 4.8173417321852545,
 170672: 4.8144468912153995,
 153976: 4.8141125664528115,
 53173: 4.811116399961962,
 121511: 4.810558966426646,
 153988: 4.810431859517403,
 32068: 4.810200918906,
 16834: 4.809378228873309,
 170446: 4.808758347367712,
 75591: 4.8084350

In [15]:
# BEGIN TRAINING DATA

train = pd.read_csv("interactions_train.csv")
train = train[["recipe_id", "rating"]]
train["recipe_id"] = train["recipe_id"].astype(int)
train["rating"] = train["rating"].astype(int)

train.head()

,recipe_id,rating
0,4684,5
1,517,5
2,7435,5
3,278,4
4,3431,5


In [16]:
# ORGANIZE ALL TRAINING RATINGS BY RECIPE
train_ratings = {}
for i in train.iterrows():
    id = i[1]["recipe_id"]
    rating = i[1]["rating"]
    if id in train_ratings:
        train_ratings[id].append(rating)
    else:
        train_ratings[id] = [rating]


In [28]:
# EXTRACTING ESSENTIAL FIELDS
train_features = data[["id","minutes","n_steps", "n_ingredients"]].copy()
train_features["id"] = train_features["id"].astype(int)
train_features["minutes"] = train_features["minutes"].astype(int)
train_features["n_steps"] = train_features["n_steps"].astype(int)
train_features["n_ingredients"] = train_features["n_ingredients"].astype(int)
train_features["ratings"] = train_features["id"].map(train_ratings)
train_features = train_features.dropna()
train_features.head()

,id,minutes,n_steps,n_ingredients,ratings
0,137739,55,11,7,"[5, 5, 5]"
1,31490,30,9,6,"[0, 4, 5]"
3,59389,45,11,11,[4]
5,5289,0,4,4,"[5, 5]"
8,70971,180,8,6,"[5, 5]"


In [37]:
train_avg_len = train_features["ratings"].apply(lambda x: len(x)).mean()
train_avg_rating = train_features["ratings"].apply(lambda x: sum(x) / len(x)).mean()

train_features["bayesian"] = train_features["ratings"].apply(lambda x: (train_avg_len * train_avg_rating + sum(x)) / (train_avg_len + len(x)))
train_features.head()

,id,minutes,n_steps,n_ingredients,ratings,bayesian
0,137739,55,11,7,"[5, 5, 5]",4.706211
1,31490,30,9,6,"[0, 4, 5]",3.889181
3,59389,45,11,11,[4],4.409115
5,5289,0,4,4,"[5, 5]",4.659898
8,70971,180,8,6,"[5, 5]",4.659898
